# Data Cleaning Polymers with Biodegradtion data

This Python Notebook is the data Cleaning polymers

- Polymer 22 data points with some of them has mutliple timepoints and relavent bidegradebilty decay

- We will find the cosine Similarty Between both two DFs and then rearrnge them into a single data frame for Further DNN Training input.

# Steps for Process TOXIC Molecule Data Sets and manomer Data set to get Cannonicalised SMILE STRINGS
- Load the Toxic Data sets (/home/sunil/am2/poetry-demo/Polytox_Matser_Thesis/Few_shot_learning_with_validation_data_Monomers/Tox_df_Category.csv)
- read the CSV File
- Drop the unwanted Columns
- Check the Current SMILES are correct or not?
- **Canonicalization** Using "RDKIT" Module
- Add both coulumns in the oroginal data frame ( Canonicalized_smiles, Vaild)
-  Count the False Values ( It should come 32)
-  List the Rows where 'is_valid' is False
- Drop the Rows with "False" Values
- Prepare the **Final Toxin + Manomer  Dataset with "4780" Datapoints with Cananonicalised correct Smile Strings**

In [2]:
import os
import subprocess
import torch

def get_filtered_free_gpus(allowed_gpus={0, 2, 3}):
    """Return (gpu_id, free_mem_MB) from allowed_gpus, sorted by free memory."""
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=index,memory.free', '--format=csv,noheader,nounits'],
            encoding='utf-8'
        )
        gpu_info = []
        for line in result.strip().split('\n'):
            gpu_id, mem_free = line.strip().split(',')
            gpu_id = int(gpu_id)
            mem_free = int(mem_free)
            if gpu_id in allowed_gpus:
                gpu_info.append((gpu_id, mem_free))
        return sorted(gpu_info, key=lambda x: x[1], reverse=True)
    except Exception as e:
        print("⚠️ Could not query GPU info:", e)
        return []

# Step 1: Get preferred GPU (among 0, 2, 3)
available_gpus = get_filtered_free_gpus()

if available_gpus:
    selected_system_gpu = available_gpus[0][0]
    os.environ["CUDA_VISIBLE_DEVICES"] = str(selected_system_gpu)
    print(f"🧠 Selected system GPU ID: {selected_system_gpu} with {available_gpus[0][1]} MB free memory.")
else:
    print("⚠️ No preferred GPUs available. Using CPU.")

# Step 2: PyTorch setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    current_device = torch.cuda.current_device()
    visible_device_name = torch.cuda.get_device_name(current_device)
    visible_ids = os.environ["CUDA_VISIBLE_DEVICES"].split(",")
    real_system_gpu_id = visible_ids[current_device]

    print("✅ GPU is available!")
    print(f"🖥️ Visible PyTorch Device: cuda:{current_device}")
    print(f"🧭 Actual system GPU ID: {real_system_gpu_id}")
    print(f"📟 GPU Name: {visible_device_name}")
else:
    print("⚠️ GPU not available. Using CPU.")

print(f"Model will run on: {device}")



🧠 Selected system GPU ID: 0 with 40323 MB free memory.
✅ GPU is available!
🖥️ Visible PyTorch Device: cuda:0
🧭 Actual system GPU ID: 0
📟 GPU Name: NVIDIA A100-SXM4-40GB
Model will run on: cuda


In [5]:
import os
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Silence parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. Verify versions
print(f"NumPy version: {np.__version__}")

NumPy version: 1.26.4


In [6]:
%pip install git+https://github.com/kuennethgroup/psmiles.git

import pandas as pd
import numpy as np
from psmiles import PolymerSmiles as PS

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/kuennethgroup/psmiles.git to /tmp/pip-req-build-qq46v9qd
  Running command git clone --filter=blob:none --quiet https://github.com/kuennethgroup/psmiles.git /tmp/pip-req-build-qq46v9qd
  Resolved https://github.com/kuennethgroup/psmiles.git to commit b5b8d3d9e7453feacb0f65aa97266425e2cb699e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy

In [7]:
# 2. Verify versions
print(f"NumPy version: {np.__version__}")

NumPy version: 1.26.4


In [ ]:
# %pip install git+https://github.com/Ramprasad-Group/psmiles.git

# # import pandas as pd
# # import numpy as np
# # from psmiles import PolymerSmiles as PS

In [8]:
from rdkit import Chem
from rdkit.Chem import AllChem
from sentence_transformers import SentenceTransformer      
polyBERT = SentenceTransformer('kuelumbus/polyBERT')

file_path = r"/home/sunil/am2/poetry-demo/AM2_Poly_biodegradebilty/Data_preprocessing_polymer/PSMILE_CO2_Biodegrade.csv"

polymer_df = pd.read_csv(file_path, encoding = "latin1")
polymer_df

2026-03-17 00:01:29.678133: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-17 00:01:30.400627: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/sunil/.local/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


,Substance Name,psmiles,Guideline,Time_day,Biodegradation
0,Polyethylene glycol,*CCO*,OECD 301B,7,0.240
1,Polyethylene glycol,*CCO*,OECD 301B,9,0.560
2,Polyethylene glycol,*CCO*,OECD 301B,11,0.610
3,Polyethylene glycol,*CCO*,OECD 301B,14,0.650
4,Polyethylene glycol,*CCO*,OECD 301B,17,0.690
...,...,...,...,...,...
75,Cellulose,*OC1C(O)C(O)C(OC1(CO))*,OECD 301B,11,0.535
76,Poly(butylene succinate),*CCCCOC(=O)CCC(=O)O*,OECD 301B,11,0.380
77,Polyamide 6,*CCCCCC(=O)N*,OECD 301B,28,0.990
78,Polycaprolactone,*CCCCCC(=O)O*,OECD 301B,5,0.039


In [9]:
from rdkit import Chem

def process_smiles(smiles):
    try:
        # Attempt to create a molecule object from the SMILES string
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            # If valid, canonicalize the SMILES
            correct_smiles = Chem.MolToSmiles(mol)
            return correct_smiles, True
        else:
            # If RDKit can't create a molecule, try to sanitize and kekulize
            mol = Chem.MolFromSmiles(smiles, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                    Chem.Kekulize(mol)
                    correct_smiles = Chem.MolToSmiles(mol)
                    return correct_smiles, True
                except:
                    pass
            # If all attempts fail, return the original SMILES and False
            return smiles, False
    except Exception as e:
        # If any error occurs in parsing, return the original SMILES and False
        return smiles, False

# Assuming your DataFrame is named `Tox_df` and the column with SMILES strings is named 'SMILES'
polymer_df['CANONICALISED_SMILES'], polymer_df['VALID'] = zip(*polymer_df['psmiles'].apply(process_smiles))

In [10]:
polymer_df

,Substance Name,psmiles,Guideline,Time_day,Biodegradation,CANONICALISED_SMILES,VALID
0,Polyethylene glycol,*CCO*,OECD 301B,7,0.240,*CCO*,True
1,Polyethylene glycol,*CCO*,OECD 301B,9,0.560,*CCO*,True
2,Polyethylene glycol,*CCO*,OECD 301B,11,0.610,*CCO*,True
3,Polyethylene glycol,*CCO*,OECD 301B,14,0.650,*CCO*,True
4,Polyethylene glycol,*CCO*,OECD 301B,17,0.690,*CCO*,True
...,...,...,...,...,...,...,...
75,Cellulose,*OC1C(O)C(O)C(OC1(CO))*,OECD 301B,11,0.535,*OC1C(CO)OC(*)C(O)C1O,True
76,Poly(butylene succinate),*CCCCOC(=O)CCC(=O)O*,OECD 301B,11,0.380,*CCCCOC(=O)CCC(=O)O*,True
77,Polyamide 6,*CCCCCC(=O)N*,OECD 301B,28,0.990,*CCCCCC(=O)N*,True
78,Polycaprolactone,*CCCCCC(=O)O*,OECD 301B,5,0.039,*CCCCCC(=O)O*,True


In [11]:
true_count = polymer_df['VALID'].value_counts().get(True, 0)
print("Number of 'True' in 'VALID':", true_count)

Number of 'True' in 'VALID': 80


In [12]:
polymer_df.shape

(80, 7)

In [13]:
# Filter the DataFrame for rows where 'VALID' is False and print them
false_rows_df = polymer_df[polymer_df['VALID'] == False]
print(false_rows_df)

Empty DataFrame
Columns: [Substance Name, psmiles, Guideline, Time_day, Biodegradation, CANONICALISED_SMILES, VALID]
Index: []


In [14]:
# 1. Find the index of rows where 'VALID' is False
rows_to_drop = polymer_df[polymer_df['VALID'] == False].index

# 2. Drop these rows by their index
polymer_df_final = polymer_df.drop(rows_to_drop)


In [15]:
print(polymer_df_final.shape)

(80, 7)


In [16]:
polymer_df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Substance Name        80 non-null     object 
 1   psmiles               80 non-null     object 
 2   Guideline             80 non-null     object 
 3   Time_day              80 non-null     int64  
 4   Biodegradation        80 non-null     float64
 5   CANONICALISED_SMILES  80 non-null     object 
 6   VALID                 80 non-null     bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 4.0+ KB


In [17]:
polymer_df_final

,Substance Name,psmiles,Guideline,Time_day,Biodegradation,CANONICALISED_SMILES,VALID
0,Polyethylene glycol,*CCO*,OECD 301B,7,0.240,*CCO*,True
1,Polyethylene glycol,*CCO*,OECD 301B,9,0.560,*CCO*,True
2,Polyethylene glycol,*CCO*,OECD 301B,11,0.610,*CCO*,True
3,Polyethylene glycol,*CCO*,OECD 301B,14,0.650,*CCO*,True
4,Polyethylene glycol,*CCO*,OECD 301B,17,0.690,*CCO*,True
...,...,...,...,...,...,...,...
75,Cellulose,*OC1C(O)C(O)C(OC1(CO))*,OECD 301B,11,0.535,*OC1C(CO)OC(*)C(O)C1O,True
76,Poly(butylene succinate),*CCCCOC(=O)CCC(=O)O*,OECD 301B,11,0.380,*CCCCOC(=O)CCC(=O)O*,True
77,Polyamide 6,*CCCCCC(=O)N*,OECD 301B,28,0.990,*CCCCCC(=O)N*,True
78,Polycaprolactone,*CCCCCC(=O)O*,OECD 301B,5,0.039,*CCCCCC(=O)O*,True


In [18]:
polymer_df_final.drop(columns=["VALID","Substance Name","Guideline","psmiles"], inplace=True)

In [19]:
polymer_df_final

,Time_day,Biodegradation,CANONICALISED_SMILES
0,7,0.240,*CCO*
1,9,0.560,*CCO*
2,11,0.610,*CCO*
3,14,0.650,*CCO*
4,17,0.690,*CCO*
...,...,...,...
75,11,0.535,*OC1C(CO)OC(*)C(O)C1O
76,11,0.380,*CCCCOC(=O)CCC(=O)O*
77,28,0.990,*CCCCCC(=O)N*
78,5,0.039,*CCCCCC(=O)O*


In [25]:
print(polymer_df_final.columns.tolist())

['Polymer name ', 'AQTOX', 'CARCINOGEN_C', 'MUTAGEN_M', 'STOT_RE', 'LOC', 'CANONICALISED_SMILES']


In [26]:
polymer_df_final.drop(columns=["Polymer name "], inplace=True)

In [20]:
polymer_df_final

,Time_day,Biodegradation,CANONICALISED_SMILES
0,7,0.240,*CCO*
1,9,0.560,*CCO*
2,11,0.610,*CCO*
3,14,0.650,*CCO*
4,17,0.690,*CCO*
...,...,...,...
75,11,0.535,*OC1C(CO)OC(*)C(O)C1O
76,11,0.380,*CCCCOC(=O)CCC(=O)O*
77,28,0.990,*CCCCCC(=O)N*
78,5,0.039,*CCCCCC(=O)O*


In [21]:
# Step 1: Rearrange the columns to move "CANONICALISED_SMILES" to the 4th position
cols = polymer_df_final.columns.tolist()  # Get the list of columns
cols.insert(0, cols.pop(cols.index('CANONICALISED_SMILES')))  # Move "CANONICALISED_SMILES" to the 1st position

# Step 2: Reassign the DataFrame with the new column order
polymer_df_final= polymer_df_final[cols]
polymer_df_final

,CANONICALISED_SMILES,Time_day,Biodegradation
0,*CCO*,7,0.240
1,*CCO*,9,0.560
2,*CCO*,11,0.610
3,*CCO*,14,0.650
4,*CCO*,17,0.690
...,...,...,...
75,*OC1C(CO)OC(*)C(O)C1O,11,0.535
76,*CCCCOC(=O)CCC(=O)O*,11,0.380
77,*CCCCCC(=O)N*,28,0.990
78,*CCCCCC(=O)O*,5,0.039


In [22]:
# 2. Verify versions
print(f"NumPy version: {np.__version__}")

NumPy version: 1.26.4


In [23]:
polymer_df_final_arr = polymer_df_final["CANONICALISED_SMILES"].to_numpy()
polymer_df_final_arr

array(['*CCO*', '*CCO*', '*CCO*', '*CCO*', '*CCO*', '*CCO*', '*CCO*',
       '*CCO*', '*CCO*', '*CCO*', '*CCO*', '*CCO*', '*CC(C)O*',
       '*CC(C)O*', '*CC(*)(C)C(=O)OC', '*CC(*)C(N)=O', '*CC(*)C(=O)O',
       '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O',
       '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O',
       '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)(O)C(=O)O', '*CCN*.*CCO*',
       '*CC(*)O', '*CCO*', '*CC(C)O*', '*CC(C)O*', '*CC(C)O*', '*CC(C)O*',
       '*CCO*', '*CCO*', '*OC1C(CO)OC(*)C(N)C1O', '*CC(C)OC(=O)O*',
       '*OC(=O)CC(*)C.*OC(=O)CC(*)CC', '*OC(=O)CC(*)C.*OC(=O)CC(*)CCC',
       '*OC(=O)CC(*)C.*OC(=O)CC(*)CC', '*OC(=O)CC(*)C.*OC(=O)CC(*)CC',
       '*NC(=O)CC(*)C(=O)O', '*NC(=O)CC(*)C(=O)O', '*NC(=O)CC(*)C(=O)O',
       '*NC(=O)CC(*)C(=O)O', '*NC(=O)CC(*)C(=O)O', '*NC(=O)CC(*)C(=O)O',
       '*NC(=O)CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O',
       '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O', '*CC(*)C(=O)O',
       '*CC(*)C(

In [24]:
polyBERT = SentenceTransformer('kuelumbus/polyBERT')
Polymer_FP= polyBERT.encode(polymer_df_final_arr)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [25]:
Polymer_FP

array([[ 0.26430973,  0.49364138, -0.28506473, ..., -0.96663487,
         0.4878483 ,  0.1659913 ],
       [ 0.26430973,  0.49364138, -0.28506473, ..., -0.96663487,
         0.4878483 ,  0.1659913 ],
       [ 0.26430973,  0.49364138, -0.28506473, ..., -0.96663487,
         0.4878483 ,  0.1659913 ],
       ...,
       [ 0.8100995 ,  0.29311335,  0.7971803 , ...,  0.27937648,
        -0.21905026,  0.00368783],
       [ 0.7145974 ,  0.75301117,  0.7734482 , ..., -0.08352884,
         0.04070525,  0.33470124],
       [ 0.7241479 ,  0.10303811,  0.48090973, ...,  0.3980961 ,
        -0.22433901,  0.24807538]], dtype=float32)

In [26]:
Polymer_FP.shape

(80, 600)

In [27]:
Polymer_FP_df =  pd.DataFrame(Polymer_FP)
Polymer_FP_df

,0,1,2,3,4,5,6,7,8,9,...,590,591,592,593,594,595,596,597,598,599
0,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,0.833980,0.109044,0.040916,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991
1,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,0.833980,0.109044,0.040916,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991
2,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,0.833980,0.109044,0.040916,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991
3,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,0.833980,0.109044,0.040916,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991
4,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253126,-1.445232,0.996869,...,0.833979,0.109044,0.040916,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.388841,0.869682,0.759189,-0.551930,0.153756,1.032118,1.780653,0.556547,0.268439,0.242710,...,0.084757,-0.443514,-0.147299,-0.296520,0.164960,-0.685685,-0.220451,-0.851011,0.515309,0.389623
76,0.512735,0.834901,0.256401,0.168272,-0.809721,-0.304924,1.084803,0.357721,-0.401167,0.957182,...,0.754353,-0.176182,0.244169,-0.182638,-0.175193,-0.844764,1.541730,-0.130815,-0.090264,0.540472
77,0.810099,0.293113,0.797180,-0.892868,0.296792,-0.145938,1.850941,0.646475,0.385395,0.792081,...,0.668064,-0.964801,-0.329933,-1.474068,-0.522120,-0.136525,1.632564,0.279376,-0.219050,0.003688
78,0.714597,0.753011,0.773448,-0.306618,-0.737555,-0.133109,1.384951,0.102281,-0.478446,0.874514,...,0.455268,-0.355176,-0.074863,-1.026752,-0.221479,-0.646499,2.337847,-0.083529,0.040705,0.334701


In [28]:
Polymer_FP_df.shape

(80, 600)

In [29]:
polymer_df_final.shape

(80, 3)

In [30]:
# Reset the index on both DataFrames
Polymer_FP_df.reset_index(drop=True, inplace=True)
polymer_df_final.reset_index(drop=True, inplace=True)

In [31]:
# Concatenate the DataFrames
polymer_DNN_data = pd.concat([Polymer_FP_df, polymer_df_final], axis=1)

In [32]:
polymer_DNN_data.shape

(80, 603)

In [33]:
polymer_DNN_data

,0,1,2,3,4,5,6,7,8,9,...,593,594,595,596,597,598,599,CANONICALISED_SMILES,Time_day,Biodegradation
0,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991,*CCO*,7,0.240
1,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991,*CCO*,9,0.560
2,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991,*CCO*,11,0.610
3,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253125,-1.445232,0.996869,...,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991,*CCO*,14,0.650
4,0.264310,0.493641,-0.285065,-0.552783,-0.874686,0.188906,1.247869,-0.253126,-1.445232,0.996869,...,-1.094582,-0.230701,-0.393534,2.199489,-0.966635,0.487848,0.165991,*CCO*,17,0.690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.388841,0.869682,0.759189,-0.551930,0.153756,1.032118,1.780653,0.556547,0.268439,0.242710,...,-0.296520,0.164960,-0.685685,-0.220451,-0.851011,0.515309,0.389623,*OC1C(CO)OC(*)C(O)C1O,11,0.535
76,0.512735,0.834901,0.256401,0.168272,-0.809721,-0.304924,1.084803,0.357721,-0.401167,0.957182,...,-0.182638,-0.175193,-0.844764,1.541730,-0.130815,-0.090264,0.540472,*CCCCOC(=O)CCC(=O)O*,11,0.380
77,0.810099,0.293113,0.797180,-0.892868,0.296792,-0.145938,1.850941,0.646475,0.385395,0.792081,...,-1.474068,-0.522120,-0.136525,1.632564,0.279376,-0.219050,0.003688,*CCCCCC(=O)N*,28,0.990
78,0.714597,0.753011,0.773448,-0.306618,-0.737555,-0.133109,1.384951,0.102281,-0.478446,0.874514,...,-1.026752,-0.221479,-0.646499,2.337847,-0.083529,0.040705,0.334701,*CCCCCC(=O)O*,5,0.039


In [34]:
# Get the first 600 columns (fingerprints) as lists
polymer_DNN_data["fingerprints"] = polymer_DNN_data.iloc[:, :600].values.tolist()

polymer_DNN_data = polymer_DNN_data.iloc[:, 600:]

polymer_DNN_data

,CANONICALISED_SMILES,Time_day,Biodegradation,fingerprints
0,*CCO*,7,0.240,"[0.26430973410606384, 0.49364137649536133, -0...."
1,*CCO*,9,0.560,"[0.26430973410606384, 0.49364137649536133, -0...."
2,*CCO*,11,0.610,"[0.26430973410606384, 0.49364137649536133, -0...."
3,*CCO*,14,0.650,"[0.26430973410606384, 0.49364137649536133, -0...."
4,*CCO*,17,0.690,"[0.26430952548980713, 0.4936414659023285, -0.2..."
...,...,...,...,...
75,*OC1C(CO)OC(*)C(O)C1O,11,0.535,"[0.3888411521911621, 0.8696816563606262, 0.759..."
76,*CCCCOC(=O)CCC(=O)O*,11,0.380,"[0.5127351880073547, 0.8349013328552246, 0.256..."
77,*CCCCCC(=O)N*,28,0.990,"[0.8100994825363159, 0.2931133508682251, 0.797..."
78,*CCCCCC(=O)O*,5,0.039,"[0.7145974040031433, 0.753011167049408, 0.7734..."


In [35]:
# Step 2: Move fingerprints column to first position
cols = polymer_DNN_data.columns.tolist()
cols.remove('fingerprints')
cols.insert(1, 'fingerprints')
polymer_DNN_data = polymer_DNN_data[cols]
polymer_DNN_data 

,CANONICALISED_SMILES,fingerprints,Time_day,Biodegradation
0,*CCO*,"[0.26430973410606384, 0.49364137649536133, -0....",7,0.240
1,*CCO*,"[0.26430973410606384, 0.49364137649536133, -0....",9,0.560
2,*CCO*,"[0.26430973410606384, 0.49364137649536133, -0....",11,0.610
3,*CCO*,"[0.26430973410606384, 0.49364137649536133, -0....",14,0.650
4,*CCO*,"[0.26430952548980713, 0.4936414659023285, -0.2...",17,0.690
...,...,...,...,...
75,*OC1C(CO)OC(*)C(O)C1O,"[0.3888411521911621, 0.8696816563606262, 0.759...",11,0.535
76,*CCCCOC(=O)CCC(=O)O*,"[0.5127351880073547, 0.8349013328552246, 0.256...",11,0.380
77,*CCCCCC(=O)N*,"[0.8100994825363159, 0.2931133508682251, 0.797...",28,0.990
78,*CCCCCC(=O)O*,"[0.7145974040031433, 0.753011167049408, 0.7734...",5,0.039


In [36]:
polymer_DNN_data.to_csv("polymer_FP.csv", index=False)

In [37]:
# Safe drop (ignores if columns don't exist)
columns_to_drop = ['CANONICALISED_SMILES']
polymer_DNN_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')

/tmp/ipykernel_807045/2329325985.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  polymer_DNN_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [38]:
polymer_DNN_data

,fingerprints,Time_day,Biodegradation
0,"[0.26430973410606384, 0.49364137649536133, -0....",7,0.240
1,"[0.26430973410606384, 0.49364137649536133, -0....",9,0.560
2,"[0.26430973410606384, 0.49364137649536133, -0....",11,0.610
3,"[0.26430973410606384, 0.49364137649536133, -0....",14,0.650
4,"[0.26430952548980713, 0.4936414659023285, -0.2...",17,0.690
...,...,...,...
75,"[0.3888411521911621, 0.8696816563606262, 0.759...",11,0.535
76,"[0.5127351880073547, 0.8349013328552246, 0.256...",11,0.380
77,"[0.8100994825363159, 0.2931133508682251, 0.797...",28,0.990
78,"[0.7145974040031433, 0.753011167049408, 0.7734...",5,0.039


In [39]:
# Define the export path
export_path = "/home/sunil/am2/poetry-demo/AM2_Poly_biodegradebilty/Data_preprocessing_polymer/POL_DNN_data.csv"

# Export to CSV
polymer_DNN_data.to_csv(export_path, index=False)

print(f"Dataset exported successfully!")
print(f"File saved to: {export_path}")
print(f"Dataset shape: {polymer_DNN_data.shape}")

Dataset exported successfully!
File saved to: /home/sunil/am2/poetry-demo/AM2_Poly_biodegradebilty/Data_preprocessing_polymer/POL_DNN_data.csv
Dataset shape: (80, 3)


In [40]:
# See all column names
print(polymer_DNN_data.columns.tolist())

['fingerprints', 'Time_day', 'Biodegradation']


In [41]:
polymer_DNN_data


,fingerprints,Time_day,Biodegradation
0,"[0.26430973410606384, 0.49364137649536133, -0....",7,0.240
1,"[0.26430973410606384, 0.49364137649536133, -0....",9,0.560
2,"[0.26430973410606384, 0.49364137649536133, -0....",11,0.610
3,"[0.26430973410606384, 0.49364137649536133, -0....",14,0.650
4,"[0.26430952548980713, 0.4936414659023285, -0.2...",17,0.690
...,...,...,...
75,"[0.3888411521911621, 0.8696816563606262, 0.759...",11,0.535
76,"[0.5127351880073547, 0.8349013328552246, 0.256...",11,0.380
77,"[0.8100994825363159, 0.2931133508682251, 0.797...",28,0.990
78,"[0.7145974040031433, 0.753011167049408, 0.7734...",5,0.039


In [43]:

# 4. Verify final structure
print(f"\n📊 FINAL DATASET STRUCTURE:")
print(f"Shape: {polymer_DNN_data.shape}")
print(f"Columns: {list(polymer_DNN_data.columns)}")



📊 FINAL DATASET STRUCTURE:
Shape: (80, 3)
Columns: ['fingerprints', 'Time_day', 'Biodegradation']


# Correct Shape: (80,3) - Clean and manageable

Clean Feature Set:

- fingerprints →  Time ( in days)
- 
 - Clear Target: Biodegrdation → in % ( ex:0.240 means 24%)
 - No Missing Data: All encoded properly

 - Ready for Model Training! 🚀
Your dataset is now perfectly prepared for:

Next Steps:
- Extract fingerprint vectors from the fingerprints column (convert from string to numerical array)
- Combine features (fingerprints + property columns)
- Train-test split
- Build DNN model
- Transfer learning implementation

The fingerprints are currently stored as strings/lists, but we need to convert them to numerical arrays and combine with the property columns